In [ ]:
from src.api import getHistoricoMOW
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
import pandas as pd
import numpy as np
from datetime import timedelta
from src.utils.topos import loadElementosTopos
from src.api.api import GraylogAPIProcessor
from src.processor import SitraProcessor
from datetime import datetime
from pathlib import Path
import csv
from src.utils.timeformat import formatTimedelta
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
    guardarExcelMulti
)
from src.utils.util import loadEstaciones
import xml.etree.ElementTree as ET
import pandas as pd
import re
import requests
import numpy as np
from requests.auth import HTTPBasicAuth
import pandas as pd
import json
from datetime import datetime, timezone, timedelta
import pickle
import os
import re
from src.utils.ficheros import guardarExcel, guardarExcelMulti
from pathlib import Path
from src.utils.util import loadEstaciones,loadEstacionSinCTC
import time
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime
)

In [ ]:

GRAYLOG_URL = "http://silog.solvan.comun.adif"
USER = "E942433"
PASSWORD = "Xiaohuanghua5"

# def search_logs(query, from_date, to_date, stream_id=None)
#     url = f"{GRAYLOG_URL}/api/search/universal/absolute"
    
#     params = {
#         "query": query,
#         "from": from_date,
#         "to": to_date,
#         "offset": 0,
#         "fields": "timestamp,source,message,level"
#     }
    
#     # Añadir filtro de stream si se especifica
#     if stream_id:
#         params["filter"] = f"streams:{stream_id}"
    
#     headers = {"Accept": "application/json"}
    
#     response = requests.get(
#         url,
#         params=params,
#         auth=HTTPBasicAuth(USER, PASSWORD),
#         headers=headers
#     )
    
#     response.raise_for_status()
#     return response.json()


# Ver streams disponibles
def get_streams():
    url = f"{GRAYLOG_URL}/api/streams"
    response = requests.get(
        url,
        auth=HTTPBasicAuth(USER, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    streams = response.json()["streams"]
    for s in streams:
        print(f"ID: {s['id']} | Nombre: {s['title']}")
    return streams



In [ ]:

def graylog_get(url, params, headers, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(
                url, params=params,
                auth=HTTPBasicAuth(USER, PASSWORD),
                headers=headers, timeout=30
            )
            if r.status_code == 500:
                raise requests.exceptions.HTTPError("500", response=r)
            r.raise_for_status()
            return r.json()

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                raise  # Propagar 500 sin reintentar — lo gestiona fetch_window
            raise

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = 2 ** attempt
            print(f"\n⏱️  Error red — reintento {attempt+1}/{max_retries} en {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"❌ Fallaron {max_retries} reintentos")


def fetch_window(query, from_str, to_str, stream_id, headers,
                 batch_size=5000, _depth=0):
    """
    Descarga una ventana de tiempo. Si encuentra error 500 por offset alto,
    divide la ventana en 2 mitades y las descarga recursivamente.
    Máximo 8 niveles de recursión (ventana mínima ~1s).
    """
    if _depth > 8:
        print(f"\n⛔ Ventana demasiado densa incluso dividida: {from_str} → {to_str}")
        return []

    messages = []
    offset    = 0

    while True:
        params = {
            "query": query, "from": from_str, "to": to_str,
            "limit": batch_size, "offset": offset,
            "fields": "timestamp,source,message,level,contentType"
        }
        if stream_id:
            params["filter"] = f"streams:{stream_id}"

        try:
            data  = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
            batch = data.get("messages", [])
            total = data.get("total_results", 0)
            messages.extend(batch)
            offset += len(batch)
            print(f"  {from_str} → {to_str} | {offset}/{total}", end="\r")
            if offset >= total or not batch:
                break

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                # Demasiados resultados con offset alto → dividir ventana en 2
                t_from = datetime.fromisoformat(from_str.replace("Z", "+00:00"))
                t_to   = datetime.fromisoformat(to_str.replace("Z", "+00:00"))
                mid    = t_from + (t_to - t_from) / 2
                mid_str = mid.strftime("%Y-%m-%dT%H:%M:%S.000Z")

                print(f"\n✂️  Dividiendo [{from_str} → {to_str}] (offset={offset}, depth={_depth})")

                # Descartar mensajes parciales de esta ventana y rehacer por mitades
                left  = fetch_window(query, from_str, mid_str, stream_id, headers,
                                     batch_size, _depth + 1)
                right = fetch_window(query, mid_str, to_str, stream_id, headers,
                                     batch_size, _depth + 1)
                return messages[:offset - len(batch)] + left + right
            raise

    return messages


def get_total_results(query, from_str, to_str, stream_id, headers):
    params = {"query": query, "from": from_str, "to": to_str,
              "limit": 1, "offset": 0, "fields": "timestamp"}
    if stream_id:
        params["filter"] = f"streams:{stream_id}"
    data = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
    return data.get("total_results", 0)


def search_logs(query, from_date, to_date, stream_id=None, limit=None,
                max_per_window=5000, checkpoint_file="checkpoint.pkl"):
    headers      = {"Accept": "application/json"}
    all_messages = []
    range_start  = datetime.fromisoformat(from_date.replace("Z", "+00:00"))
    range_end    = datetime.fromisoformat(to_date.replace("Z", "+00:00"))
    total_seconds = (range_end - range_start).total_seconds()

    # Reanudar desde checkpoint
    resume_from = range_start
    if os.path.exists(checkpoint_file):
        print(f"♻️  Reanudando desde checkpoint...")
        with open(checkpoint_file, "rb") as f:
            ckpt = pickle.load(f)
        all_messages = ckpt["messages"]
        resume_from  = ckpt["last_window_end"]
        print(f"   {len(all_messages):,} msgs ya descargados, continuando desde {resume_from}\n")

    # Calcular ventana óptima
    print("🔍 Calculando total de mensajes...")
    total_global = get_total_results(
        query,
        range_start.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        range_end.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        stream_id, headers
    )
    print(f"📊 Total: {total_global:,}")

    if total_global == 0:
        return []

    density        = total_global / total_seconds
    window_seconds = max(int((max_per_window / density) * 0.85), 5)
    remaining      = (range_end - resume_from).total_seconds()
    print(f"⚙️  Ventana: {window_seconds}s | Estimadas: {int(remaining/window_seconds)+1}\n")

    current      = resume_from
    window_count = 0

    while current < range_end:
        window_end = min(current + timedelta(seconds=window_seconds), range_end)
        from_str   = current.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        to_str     = window_end.strftime("%Y-%m-%dT%H:%M:%S.000Z")

        try:
            batch = fetch_window(query, from_str, to_str, stream_id, headers)
            all_messages.extend(batch)
            window_count += 1
            print(f"✅ {from_str} → {to_str} | +{len(batch):,} | Total: {len(all_messages):,}")

        except RuntimeError as e:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": current}, f)
            print(f"\n💾 Guardado emergencia: {len(all_messages):,} msgs")
            raise e

        # Checkpoint cada 50 ventanas
        if window_count % 50 == 0:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": window_end}, f)
            print(f"💾 Checkpoint: {len(all_messages):,} msgs")

        current = window_end

        if limit and len(all_messages) >= limit:
            all_messages = all_messages[:limit]
            print(f"\n🛑 Límite alcanzado: {limit:,} msgs")
            break

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"\n✅ Descarga completa: {len(all_messages):,} mensajes")
    return all_messages


def search_logs_to_dataframe(query, from_date, to_date, stream_id=None,
                              limit=None, checkpoint_file="checkpoint.pkl"):
    messages = search_logs(query, from_date, to_date, stream_id, limit,
                           checkpoint_file=checkpoint_file)
    if not messages:
        return pd.DataFrame()
    rows = [msg.get("message", msg) for msg in messages]
    return pd.DataFrame(rows)

In [ ]:
stream = get_streams()

In [ ]:
from datetime import datetime, timedelta, timezone

hoy = datetime.now(timezone.utc)
anteayer = hoy - timedelta(days=2)

data = search_logs(
    query="originChange",
    from_date="2026-05-04T00:00:00.000Z",
    to_date="2026-05-11T00:00:00.000Z",
    stream_id="692d53016456d79315fb46c6 ",
    limit=None
)


In [ ]:


def parse_originchange_messages(data):
    rows = []
    for item in data:
        msg_dict = item.get("message", {})
        raw = msg_dict.get("message", "")
        graylog_ts = msg_dict.get("timestamp", "")
        source     = msg_dict.get("source", "")

        # Extraer el bloque XML del mensaje
        xml_match = re.search(r"(<ruOperationRequest>.*?</ruOperationRequest>)", raw, re.DOTALL)
        if not xml_match:
            continue

        try:
            root = ET.fromstring(xml_match.group(1))
            row = {child.tag: child.text for child in root}
            row["graylog_timestamp"] = graylog_ts
            row["source"]            = source
            rows.append(row)
        except ET.ParseError as e:
            print(f"⚠️ XML inválido en {source}: {e}")

    return pd.DataFrame(rows)


df = parse_originchange_messages(data)

In [ ]:
df["productCode"].unique()

In [ ]:
df

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:

def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    xSIVPLUS:bool = False,
    jCTC: bool = False,
    xREG: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        xSIVPLUS = xSIVPLUS,
        jCTC=jCTC,
        xREG=xREG,
        pro=pro,
        maniobra= maniobra
    )
        
    if (xREG == False): 
        historico = historico[
            (historico["Fecha"] >= pd.to_datetime(start_date))
            & (historico["Fecha"] <= pd.to_datetime(end_date))
        ]
    else:
         historico = historico[
            (historico["FechaHora"] >= pd.to_datetime(start_date))
            & (historico["FechaHora"] <= pd.to_datetime(end_date))
        ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    
    
    
    if (xREG == False): 
        historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
            subset=["Movimiento"]
        )
        historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2026-05-04"
end_date = "2026-05-05"
estaciones = []

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    xSIVPLUS=False,
    jCTC=False,
    pro=True,
)
# historico_pro = historico_pro.sort_values(
#     by=["FechaOrigen", "NTécnico", "Fecha"]
# ).reset_index(drop=True)

# # Añadir información de la fecha
# historico_pro["Día"] = historico_pro["Fecha"].dt.date
# historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
# historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
# historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
nucleo = historico_pro[["NTécnico","Núcleo"]].copy()

In [ ]:
nucleo.drop_duplicates(subset="NTécnico",inplace=True)

In [ ]:
df_nucleo= pd.merge(
    df,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
df_nucleo

In [ ]:
import xml.etree.ElementTree as ET

def parse_sad_xml(xml_path):
    """
    Parsea sad.xml y devuelve un dict:
    { service_code (int) -> comercial_association (code_1 | code_2) }
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    mapping = {}
    for service in root.findall("service"):
        ident = service.find("identificator")
        if ident is None:
            continue

        code = ident.get("code")  # ej: "00040"
        if code is None:
            continue

        # Buscar comercial_association dentro de cualquier regulación
        for reg in service.findall(".//comercial_association"):
            code_1 = reg.get("code_1", "")
            code_2 = reg.get("code_2", "")
            # Guardar ambos; si hay varios, el primero gana (ajusta si necesitas otro criterio)
            mapping[int(code)] = {"code_1": code_1, "code_2": code_2}
            break  # primera regulación es suficiente

    return mapping

fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20260513033513.xml")
sad_mapping = parse_sad_xml(fname)


In [ ]:

# Aplicar a df donde productCode es 'M' o 'L'
mask_ml = df_nucleo["productCode"].isin(["M", "L"])

def get_nucleo_from_sad(row, mapping):
    rn = int(row["runningNumber"])
    entry = mapping.get(rn)
    if entry:
        # Devuelve code_1; cámbialo a code_2 o f"{code_1}/{code_2}" si prefieres
        return entry["code_1"]
    return None

df_nucleo.loc[mask_ml, "Núcleo"] = df_nucleo.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
)


In [ ]:
df_nucleo


<h1> DestinoChanged </h1>

In [ ]:
data_destino = search_logs(
    query="destinationChange ",
    from_date="2026-05-04T00:00:00.000Z",
    to_date="2026-05-11T00:00:00.000Z",
    stream_id="692d53016456d79315fb46c6 ",
    limit=None
)


In [ ]:
df_destino = parse_originchange_messages(data_destino)

In [ ]:
df_nucleo_destino= pd.merge(
    df_destino,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_nucleo_destino["productCode"].isin(["M", "L"])
df_nucleo_destino.loc[mask_ml, "Núcleo"] = df_nucleo_destino.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
)


In [ ]:
df_nucleo_destino

<h1> interrupción entre dos puntos</h1>

In [ ]:
data_interrupción = search_logs(
    query="interruptionBetweenTwoPoints",
    from_date="2026-05-04T00:00:00.000Z",
    to_date="2026-05-11T00:00:00.000Z",
    stream_id="692d53016456d79315fb46c6 ",
    limit=None
)

In [ ]:
df_interrupción = parse_originchange_messages(data_interrupción)

In [ ]:
df_nucleo_interrupción= pd.merge(
    df_interrupción,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_nucleo_interrupción["productCode"].isin(["M", "L"])
df_nucleo_interrupción.loc[mask_ml, "Núcleo"] = df_nucleo_interrupción.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
)


<h1>Intermediate</h1>

In [ ]:
data_intermediate = search_logs(
    query="intermediateOriginByIncidence",
    from_date="2026-05-04T00:00:00.000Z",
    to_date="2026-05-11T00:00:00.000Z",
    stream_id="692d53016456d79315fb46c6 ",
    limit=None
)

In [ ]:
df_intermediate = parse_originchange_messages(data_intermediate)

In [ ]:
df_nucleo_intermediate= pd.merge(
    df_intermediate,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_nucleo_intermediate["productCode"].isin(["M", "L"])
df_nucleo_intermediate.loc[mask_ml, "Núcleo"] = df_nucleo_intermediate.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
)


In [ ]:
df_total = pd.concat([df_nucleo,df_nucleo_destino,df_nucleo_intermediate,df_nucleo_interrupción],ignore_index=True)

<h1> SP </h1>

In [ ]:
data_sp= search_logs(
    query="movementType:SP OR movementType:SG",
    from_date="2026-05-04T00:00:00.000Z",
    to_date="2026-05-11T00:00:00.000Z",
    stream_id="692d53016456d79315fb46c6 ",
    limit=None
)

In [ ]:
def parse_realmovement_messages(data):
    rows = []
    for item in data:
        msg_dict = item.get("message", {})
        raw      = msg_dict.get("message", "")

        ts_match  = re.match(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d+)", raw)
        xml_match = re.search(r"(<realMovement>.*?</realMovement>)", raw, re.DOTALL)

        if not xml_match:
            continue
        try:
            root = ET.fromstring(xml_match.group(1))
            row  = {child.tag: child.text for child in root}
            row["graylog_timestamp"] = msg_dict.get("timestamp")
            row["log_timestamp"]     = ts_match.group(1).replace(",", ".") if ts_match else None
            row["source_host"]       = msg_dict.get("source")
            row["index"]             = item.get("index")
            rows.append(row)
        except ET.ParseError as e:
            print(f"⚠️ XML inválido: {e}")

    df = pd.DataFrame(rows)
    for col in ("log_timestamp", "graylog_timestamp"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    return df


df_sp = parse_realmovement_messages(data_sp)


In [ ]:
df_supresión = df_sp[df_sp["movementType"] == "SP"].copy()

In [ ]:
df_guadiana = df_sp[df_sp["movementType"] == "SG"].copy()

In [ ]:
df_supresión_origen = df_supresión[df_supresión["sequence"] == "001"].copy()


In [ ]:

keys = ["runningNumber", "runningDate"]

df_cancelado = df_supresión_origen[
    ~df_supresión_origen.set_index(keys).index.isin(
        df_guadiana.set_index(keys).index
    )
].reset_index(drop=True)

In [ ]:


df_sup_origen= df_supresión_origen[
    df_supresión_origen.set_index(keys).index.isin(
        df_guadiana.set_index(keys).index
    )
].reset_index(drop=True)

In [ ]:
df_supresión_no_origen = df_supresión[df_supresión["sequence"] != "001"].copy()

In [ ]:
df_sup_destino= df_supresión_no_origen[
    ~df_supresión_no_origen.set_index(keys).index.isin(
        df_guadiana.set_index(keys).index
    )
].reset_index(drop=True)

In [ ]:
df_sup_intermedio= df_supresión_no_origen[
    df_supresión_no_origen.set_index(keys).index.isin(
        df_guadiana.set_index(keys).index
    )
].reset_index(drop=True)

In [ ]:
df_cancelado_nucleo= pd.merge(
    df_cancelado,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_cancelado_nucleo["productCode"].isin(["M", "L"])
df_cancelado_nucleo.loc[mask_ml, "Núcleo"] = df_cancelado_nucleo.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
)


In [ ]:
df_cancelado_sp_origen= pd.merge(
    df_sup_origen,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_cancelado_sp_origen["productCode"].isin(["M", "L"])
df_cancelado_sp_origen.loc[mask_ml, "Núcleo"] = df_cancelado_sp_origen.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
    )


In [ ]:
df_cancelado_sup_destino = pd.merge(
    df_sup_destino,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left"
)

In [ ]:
mask_ml = df_cancelado_sup_destino["productCode"].isin(["M", "L"])
df_cancelado_sup_destino.loc[mask_ml, "Núcleo"] = df_cancelado_sup_destino.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
    )


In [ ]:
df_cancelado_sup_intermedio = pd.merge(
    df_sup_intermedio,
    nucleo,
    left_on = "runningNumber",
    right_on= "NTécnico",
    how = "left")

In [ ]:
mask_ml = df_cancelado_sup_intermedio["productCode"].isin(["M", "L"])
df_cancelado_sup_intermedio.loc[mask_ml, "Núcleo"] = df_cancelado_sup_intermedio.loc[mask_ml].apply(
    lambda row: get_nucleo_from_sad(row, sad_mapping), axis=1
    )


In [ ]:
df_cancelado_nucleo.drop(columns="NTécnico",inplace=True)

In [ ]:
df_cancelado_sp_origen.drop(columns="NTécnico",inplace=True)

In [ ]:
df_cancelado_sup_destino.drop(columns="NTécnico",inplace=True)

In [ ]:
df_cancelado_sup_intermedio.drop(columns="NTécnico",inplace=True)

In [ ]:
df_total['runningDate'] = pd.to_datetime(df_total['runningDate'].str[:10]).dt.strftime('%Y-%m-%d')

In [ ]:
df_cancelado_nucleo['runningDate'] = pd.to_datetime(df_cancelado_nucleo['runningDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [ ]:
df_cancelado_sp_origen['runningDate'] = pd.to_datetime(df_cancelado_sp_origen['runningDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [ ]:
df_cancelado_sup_destino['runningDate'] = pd.to_datetime(df_cancelado_sup_destino['runningDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [ ]:
df_cancelado_sup_intermedio['runningDate'] = pd.to_datetime(df_cancelado_sup_intermedio['runningDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [ ]:
keys_total = set(zip(df_total['runningNumber'], df_total['runningDate']))

def excluir_si_en_total(df):
    mask = [
        (r, d) in keys_total
        for r, d in zip(df['runningNumber'], df['runningDate'])
    ]
    return df[~pd.Series(mask, index=df.index)]

df_cancelado_sp_origen      = excluir_si_en_total(df_cancelado_sp_origen)
df_cancelado_sup_destino    = excluir_si_en_total(df_cancelado_sup_destino)
df_cancelado_sup_intermedio = excluir_si_en_total(df_cancelado_sup_intermedio)

In [ ]:
import pandas as pd
import json
from pathlib import Path

# ─────────────────────────────────────────────
# 1.  CONFIGURA TUS DATAFRAMES AQUÍ
# ─────────────────────────────────────────────
df_ru                    = df_total
df_one                   = df_cancelado_nucleo
df_two                   = df_cancelado_sp_origen
df_three                 = df_cancelado_sup_destino
df_four                  = df_cancelado_sup_intermedio

DATAFRAMES = {
    "RuOperationRequest":       df_ru,
    "Circulaciones Canceladas": df_one,
    "Cambio Origen":            df_two,
    "Cambio Destino":           df_three,
    "Cambio Tramo Intermedio":  df_four,
}

# ─────────────────────────────────────────────
# 2.  RECOPILAR DATOS PARA EL HTML
# ─────────────────────────────────────────────
records        = []
all_products   = set()
all_nucleos    = set()
all_operations = set()

RU_NAME = "RuOperationRequest"

for name, df in DATAFRAMES.items():
    pc_col  = "productCode" if "productCode" in df.columns else None
    nuc_col = "Núcleo"      if "Núcleo"      in df.columns else None
    op_col  = "operation"   if "operation"   in df.columns else None

    if pc_col:
        all_products.update(df[pc_col].dropna().unique())
    if nuc_col:
        all_nucleos.update(df[nuc_col].dropna().unique())
    if op_col and name == RU_NAME:
        all_operations.update(df[op_col].dropna().unique())

    # ── combos productCode × Núcleo (usados por los DFs no-RU y fallback RU) ──
    combos = []
    if pc_col and nuc_col:
        grp = df.groupby([pc_col, nuc_col], dropna=False).size().reset_index(name="count")
        for _, row in grp.iterrows():
            combos.append({"productCode": str(row[pc_col]), "nucleo": str(row[nuc_col]), "count": int(row["count"])})
    elif pc_col:
        grp = df.groupby(pc_col, dropna=False).size().reset_index(name="count")
        for _, row in grp.iterrows():
            combos.append({"productCode": str(row[pc_col]), "nucleo": "—", "count": int(row["count"])})
    elif nuc_col:
        grp = df.groupby(nuc_col, dropna=False).size().reset_index(name="count")
        for _, row in grp.iterrows():
            combos.append({"productCode": "—", "nucleo": str(row[nuc_col]), "count": int(row["count"])})
    else:
        combos.append({"productCode": "—", "nucleo": "—", "count": len(df)})

    # ── op_combos: productCode × Núcleo × operation (solo RU) ──
    op_combos  = []
    op_summary = []
    if name == RU_NAME and op_col:
        group_cols = [c for c in [pc_col, nuc_col, op_col] if c]
        grp = df.groupby(group_cols, dropna=False).size().reset_index(name="count")
        for _, row in grp.iterrows():
            op_combos.append({
                "productCode": str(row[pc_col])  if pc_col  else "—",
                "nucleo":      str(row[nuc_col]) if nuc_col else "—",
                "operation":   str(row[op_col]),
                "count":       int(row["count"]),
            })
        # conteo global por operation (sin filtros)
        op_counts = df.groupby(op_col, dropna=False).size().reset_index(name="count")
        op_summary = [
            {"operation": str(r[op_col]), "count": int(r["count"])}
            for _, r in op_counts.sort_values("count", ascending=False).iterrows()
        ]

    records.append({
        "name":        name,
        "isRu":        name == RU_NAME,
        "total":       len(df),
        "combos":      combos,
        "op_combos":   op_combos,
        "op_summary":  op_summary,
        "ru_operation": None,
        "ru_combos":    [],
    })

# ── Mapeo: nombre del DF → operation en RU que le corresponde ──
RU_OP_MAP = {
    "Cambio Origen":           "originChange",
    "Cambio Destino":          "destinationChange",
    "Cambio Tramo Intermedio": "interruptionBetweenTwoPoints",
}

ru_record = next(r for r in records if r["isRu"])
for rec in records:
    op = RU_OP_MAP.get(rec["name"])
    if op:
        rec["ru_operation"] = op
        rec["ru_combos"] = [c for c in ru_record["op_combos"] if c["operation"] == op]

all_products   = sorted(all_products)
all_nucleos    = sorted(all_nucleos)
all_operations = sorted(all_operations)

data_json      = json.dumps(records,        ensure_ascii=False)
products_json  = json.dumps(all_products,   ensure_ascii=False)
nucleos_json   = json.dumps(all_nucleos,    ensure_ascii=False)
ops_json       = json.dumps(all_operations, ensure_ascii=False)

# ─────────────────────────────────────────────
# 3.  PLANTILLA HTML
# ─────────────────────────────────────────────
HTML = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Dashboard DataFrames</title>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet"/>
<style>
  :root {{
    --bg:      #0d0f14;
    --panel:   #141720;
    --border:  #252a38;
    --accent:  #00e5a0;
    --accent2: #3d8ef8;
    --warn:    #f5a623;
    --text:    #e4e8f0;
    --muted:   #6b7594;
  }}
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{
    background: var(--bg);
    color: var(--text);
    font-family: 'IBM Plex Sans', sans-serif;
    min-height: 100vh;
  }}

  header {{
    padding: 2rem 2.5rem 1.5rem;
    border-bottom: 1px solid var(--border);
    display: flex; align-items: baseline; gap: 1rem;
  }}
  header h1 {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: 1.25rem; color: var(--accent); letter-spacing: .08em;
  }}
  header span {{ font-size: .8rem; color: var(--muted); font-family: 'IBM Plex Mono', monospace; }}

  .filters {{
    display: flex; gap: 1.2rem;
    padding: 1.2rem 2.5rem;
    background: var(--panel);
    border-bottom: 1px solid var(--border);
    flex-wrap: wrap; align-items: flex-end;
  }}
  .filter-group {{ display: flex; flex-direction: column; gap: .35rem; }}
  label {{
    font-size: .7rem; text-transform: uppercase;
    letter-spacing: .12em; color: var(--muted);
    font-family: 'IBM Plex Mono', monospace;
  }}
  select {{
    background: var(--bg); border: 1px solid var(--border);
    color: var(--text); padding: .45rem .85rem;
    border-radius: 4px; font-size: .85rem;
    font-family: 'IBM Plex Mono', monospace;
    cursor: pointer; outline: none;
    transition: border-color .2s; min-width: 180px;
  }}
  select:focus {{ border-color: var(--accent); }}
  #selOperation {{ border-color: #3a3020; color: var(--warn); }}
  #selOperation:focus {{ border-color: var(--warn); }}

  .filter-sep {{
    width: 1px; height: 36px; background: var(--border);
    align-self: flex-end; margin: 0 .4rem;
  }}
  .filter-section-label {{
    font-size: .65rem; color: var(--muted);
    font-family: 'IBM Plex Mono', monospace;
    letter-spacing: .1em; align-self: flex-end;
    padding-bottom: .6rem;
  }}
  .filter-section-label.ru {{ color: #6b5a3a; }}

  .btn-reset {{
    background: transparent; border: 1px solid var(--border);
    color: var(--muted); padding: .45rem 1rem;
    border-radius: 4px; font-size: .75rem;
    font-family: 'IBM Plex Mono', monospace;
    cursor: pointer; letter-spacing: .08em;
    transition: all .2s; align-self: flex-end;
  }}
  .btn-reset:hover {{ border-color: var(--accent); color: var(--accent); }}

  .summary {{
    padding: 1.2rem 2.5rem;
    display: flex; gap: .6rem; flex-wrap: wrap; align-items: center;
    font-family: 'IBM Plex Mono', monospace; font-size: .78rem;
    color: var(--muted); border-bottom: 1px solid var(--border);
  }}
  .summary strong {{ color: var(--accent); font-size: 1rem; }}

  .grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(300px, 1fr));
    gap: 1.2rem; padding: 2rem 2.5rem;
  }}

  .card {{
    background: var(--panel); border: 1px solid var(--border);
    border-radius: 8px; padding: 0;
    display: flex; flex-direction: column;
    transition: border-color .25s, transform .2s;
    position: relative; overflow: hidden;
    max-height: 480px;
  }}
  .card-header {{
    padding: 1.4rem 1.4rem .8rem;
    display: flex; flex-direction: column; gap: .8rem;
    flex-shrink: 0;
  }}
  .card-body {{
    overflow-y: auto;
    padding: 0 1.4rem 1.4rem;
    display: flex; flex-direction: column; gap: .8rem;
    flex: 1;
  }}
  .card-body::-webkit-scrollbar {{ width: 3px; }}
  .card-body::-webkit-scrollbar-thumb {{ background: var(--border); }}
  .card::before {{
    content: ''; position: absolute;
    top: 0; left: 0; right: 0; height: 3px;
    background: var(--accent); opacity: 0; transition: opacity .25s;
  }}
  .card:hover {{ border-color: var(--accent); transform: translateY(-2px); }}
  .card:hover::before {{ opacity: 1; }}
  .ru-card {{ border-color: #3a3020; }}
  .ru-card::before {{ background: var(--warn); opacity: 1; }}
  .ru-card:hover {{ border-color: var(--warn); }}

  .card-name {{
    font-family: 'IBM Plex Mono', monospace; font-size: .8rem;
    color: var(--muted); text-transform: uppercase;
    letter-spacing: .1em; display: flex; align-items: center; gap: .5rem;
  }}
  .badge-ru {{
    background: #3a3020; color: var(--warn);
    font-size: .65rem; padding: .1rem .4rem;
    border-radius: 3px; letter-spacing: .08em;
  }}
  .card-count {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: 2.8rem; font-weight: 600; line-height: 1; color: var(--text);
  }}
  .card-count span {{
    font-size: .7rem; color: var(--muted);
    font-weight: 400; margin-left: .4rem; vertical-align: middle;
  }}
  .bar-track {{
    height: 4px; background: var(--border); border-radius: 2px; overflow: hidden;
  }}
  .bar-fill {{
    height: 100%; background: var(--accent2);
    border-radius: 2px; transition: width .5s ease;
  }}
  .ru-card .bar-fill {{ background: var(--warn); }}

  /* desglose product × núcleo */
  .breakdown {{
    font-size: .74rem; font-family: 'IBM Plex Mono', monospace;
    color: var(--muted); border-top: 1px solid var(--border);
    padding-top: .7rem; display: flex; flex-direction: column;
    gap: .25rem; max-height: 130px; overflow-y: auto;
  }}
  .breakdown::-webkit-scrollbar {{ width: 3px; }}
  .breakdown::-webkit-scrollbar-thumb {{ background: var(--border); }}
  .brow {{
    display: grid; grid-template-columns: 56px 1fr 48px;
    gap: 0; padding: .15rem .2rem; border-radius: 3px;
  }}
  .brow:hover {{ background: rgba(255,255,255,.03); }}
  .brow > span:nth-child(1) {{ color: var(--muted); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .brow > span:nth-child(2) {{ color: var(--muted); padding-left: .5rem; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .brow .cnt {{ color: var(--text); text-align: right; white-space: nowrap; }}

  /* desglose operations (RU) */
  .op-breakdown {{
    font-size: .74rem; font-family: 'IBM Plex Mono', monospace;
    color: var(--muted); border-top: 1px solid #3a3020;
    padding-top: .7rem; display: flex; flex-direction: column;
    gap: .25rem; max-height: 130px; overflow-y: auto;
  }}
  .op-breakdown::-webkit-scrollbar {{ width: 3px; }}
  .op-breakdown::-webkit-scrollbar-thumb {{ background: #3a3020; }}
  .op-label {{
    font-size: .65rem; color: #6b5a3a;
    text-transform: uppercase; letter-spacing: .1em; margin-bottom: .1rem;
  }}
  .op-row {{
    display: grid; grid-template-columns: 1fr auto;
    gap: .4rem; padding: .15rem .2rem; border-radius: 3px;
  }}
  .op-row:hover {{ background: rgba(245,166,35,.05); }}
  .op-row .cnt {{ color: var(--warn); text-align: right; }}

  /* bloque RU dentro de cards no-RU */
  .ru-ref-block {{
    font-size: .74rem; font-family: 'IBM Plex Mono', monospace;
    border-top: 1px solid #3a3020;
    padding-top: .7rem; display: flex; flex-direction: column;
    gap: .25rem; max-height: 130px; overflow-y: auto;
  }}
  .ru-ref-block::-webkit-scrollbar {{ width: 3px; }}
  .ru-ref-block::-webkit-scrollbar-thumb {{ background: #3a3020; }}
  .ru-ref-label {{
    font-size: .65rem; color: #6b5a3a;
    text-transform: uppercase; letter-spacing: .1em;
    margin-bottom: .1rem; display: flex; align-items: center; gap: .4rem;
  }}
  .ru-ref-label .badge-ru-sm {{
    background: #3a3020; color: var(--warn);
    font-size: .6rem; padding: .05rem .35rem;
    border-radius: 3px; letter-spacing: .06em;
  }}
  .ru-ref-row {{
    display: grid; grid-template-columns: 56px 1fr 48px;
    gap: 0; padding: .15rem .2rem; border-radius: 3px;
  }}
  .ru-ref-row:hover {{ background: rgba(245,166,35,.05); }}
  .ru-ref-row > span:nth-child(1) {{ color: #8a7050; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .ru-ref-row > span:nth-child(2) {{ color: #8a7050; padding-left: .5rem; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .ru-ref-row .cnt {{ color: var(--warn); text-align: right; white-space: nowrap; }}

  .section-header {{
    display: flex; align-items: baseline;
    justify-content: space-between;
    padding: .4rem 0 .2rem;
    border-top: 1px solid var(--border);
    margin-top: .2rem;
  }}
  .section-title {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: .8rem; color: var(--muted);
    text-transform: uppercase; letter-spacing: .1em;
  }}
  .section-big-count {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: 1.6rem; font-weight: 600; color: var(--text); line-height: 1;
  }}

  .no-data {{
    text-align: center; padding: 3rem; color: var(--muted);
    font-family: 'IBM Plex Mono', monospace; font-size: .85rem;
    grid-column: 1/-1;
  }}
</style>
</head>
<body>

<header>
  <h1>▸ DataFrame Monitor</h1>
  <span id="ts"></span>
</header>

<div class="filters">
  <span class="filter-section-label">Global</span>
  <div class="filter-group">
    <label>ProductCode</label>
    <select id="selProduct"><option value="">— todos —</option></select>
  </div>
  <div class="filter-group">
    <label>Núcleo</label>
    <select id="selNucleo"><option value="">— todos —</option></select>
  </div>

  <button class="btn-reset" onclick="resetFilters()">⟳ Reset</button>
</div>

<div class="summary">
  <span>Total filas visibles:</span>
  <strong id="totalRows">0</strong>
  <span style="margin-left:.5rem">en</span>
  <strong id="totalDfs">0</strong>
  <span>DataFrames</span>
</div>

<div class="grid" id="grid"></div>

<script>
const RAW_DATA       = {data_json};
const ALL_PRODUCTS   = {products_json};
const ALL_NUCLEOS    = {nucleos_json};
const ALL_OPERATIONS = {ops_json};

const selP = document.getElementById('selProduct');
const selN = document.getElementById('selNucleo');
function addOpt(sel, v) {{
  const o = document.createElement('option');
  o.value = v; o.textContent = v; sel.appendChild(o);
}}
ALL_PRODUCTS.forEach(v => addOpt(selP, v));
ALL_NUCLEOS.forEach(v  => addOpt(selN, v));

selP.addEventListener('change', render);
selN.addEventListener('change', render);
function resetFilters() {{ selP.value=''; selN.value=''; render(); }}

// ── contar filas de un df con los filtros activos ──
function countDf(df, fp, fn, fo) {{
  if (df.isRu) {{
    return df.op_combos
      .filter(r =>
        (!fp || r.productCode === fp) &&
        (!fn || r.nucleo      === fn) &&
        (!fo || r.operation   === fo))
      .reduce((s, r) => s + r.count, 0);
  }}
  return df.combos
    .filter(r => (!fp || r.productCode === fp) && (!fn || r.nucleo === fn))
    .reduce((s, r) => s + r.count, 0);
}}

// ── colapsar a productCode × Núcleo ──
function collapseRows(rows) {{
  const map = {{}};
  rows.forEach(r => {{
    const k = r.productCode + '||' + r.nucleo;
    map[k] = (map[k] || 0) + r.count;
  }});
  return Object.entries(map).map(([k, v]) => {{
    const [pc, nc] = k.split('||');
    return {{ productCode: pc, nucleo: nc, count: v }};
  }});
}}

// ── conteo RU ref para cards no-RU (filtrado por product/nucleo) ──
function buildRuRefRows(df, fp, fn) {{
  if (!df.ru_combos || !df.ru_combos.length) return [];
  const map = {{}};
  df.ru_combos
    .filter(r => (!fp || r.productCode === fp) && (!fn || r.nucleo === fn))
    .forEach(r => {{
      const k = r.productCode + '||' + r.nucleo;
      map[k] = (map[k] || 0) + r.count;
    }});
  return Object.entries(map).map(([k, v]) => {{
    const [pc, nc] = k.split('||');
    return {{ productCode: pc, nucleo: nc, count: v }};
  }});
}}

// ── conteo por operation respetando todos los filtros (solo RU) ──
function buildOpRows(df, fp, fn, fo) {{
  const map = {{}};
  df.op_combos
    .filter(r =>
      (!fp || r.productCode === fp) &&
      (!fn || r.nucleo      === fn) &&
      (!fo || r.operation   === fo))
    .forEach(r => {{ map[r.operation] = (map[r.operation] || 0) + r.count; }});
  return Object.entries(map)
    .map(([op, cnt]) => ({{ operation: op, count: cnt }}))
    .sort((a, b) => b.count - a.count);
}}

function render() {{
  const fp = selP.value, fn = selN.value, fo = '';
  const grid = document.getElementById('grid');
  grid.innerHTML = '';

  let grandTotal = 0, dfCount = 0;
  const maxTotal = Math.max(1, ...RAW_DATA.map(df => countDf(df, fp, fn, fo)));

  RAW_DATA.forEach(df => {{
    const total = countDf(df, fp, fn, fo);
    if (total === 0 && (fp || fn || fo)) return;
    grandTotal += total; dfCount++;

    const pct  = Math.round(total / maxTotal * 100);
    const card = document.createElement('div');
    card.className = 'card' + (df.isRu ? ' ru-card' : '');

    // filas para el desglose product × núcleo
    const sourceRows = df.isRu
      ? df.op_combos.filter(r =>
          (!fp || r.productCode === fp) &&
          (!fn || r.nucleo      === fn) &&
          (!fo || r.operation   === fo))
      : df.combos.filter(r =>
          (!fp || r.productCode === fp) &&
          (!fn || r.nucleo      === fn));
    const collapsed = collapseRows(sourceRows);

    const mainRowsHtml = collapsed.map(r =>
      `<div class="brow">
         <span>${{r.productCode}}</span>
         <span>${{r.nucleo}}</span>
         <span class="cnt">${{r.count.toLocaleString('es-ES')}}</span>
       </div>`
    ).join('');

    // bloque de operations (solo RU)
    let opBlock = '';
    if (df.isRu) {{
      const opRows = buildOpRows(df, fp, fn, fo);
      const opRowsHtml = opRows.map(r =>
        `<div class="op-row">
           <span>${{r.operation}}</span>
           <span class="cnt">${{r.count.toLocaleString('es-ES')}}</span>
         </div>`
      ).join('');
      opBlock = `
        <div class="op-breakdown">
          <div class="op-label">por operation</div>
          ${{opRowsHtml || '<span style="color:var(--muted)">sin datos</span>'}}
        </div>`;
    }}

    // bloque RU ref (para cards no-RU con operación vinculada)
    let ruRefBlock = '';
    if (!df.isRu && df.ru_operation) {{
      const ruRows = buildRuRefRows(df, fp, fn);
      const ruTotal = ruRows.reduce((s, r) => s + r.count, 0);
      const ruRowsHtml = ruRows.map(r =>
        `<div class="brow">
           <span>${{r.productCode}}</span>
           <span>${{r.nucleo}}</span>
           <span class="cnt">${{r.count.toLocaleString('es-ES')}}</span>
         </div>`
      ).join('');
      ruRefBlock = `
        <div class="breakdown">
          <div class="section-header"><span class="section-title">Con petición</span><span class="section-big-count">${{ruTotal.toLocaleString('es-ES')}}</span></div>
          <div class="brow" style="color:var(--muted);font-size:.65rem;padding-bottom:.25rem;border-bottom:1px solid var(--border)">
            <span>PRODUCT</span><span>NÚCLEO</span><span class="cnt">#</span>
          </div>
          ${{ruRowsHtml || '<span style="color:var(--muted)">sin datos</span>'}}
        </div>`;
    }}

    // etiqueta "sin petición" solo en cards con bloque RU
    const sinTotal = collapsed.reduce((s,r)=>s+r.count,0);
    const sinPeticionLabel = (!df.isRu && df.ru_operation)
      ? `<div class="section-header"><span class="section-title">Sin petición</span><span class="section-big-count">${{sinTotal.toLocaleString('es-ES')}}</span></div>`
      : '';

    const hasPeticion = !df.isRu && df.ru_operation;
    card.innerHTML = `
      <div class="card-header">
        <div class="card-name">
          ${{df.name}}
          ${{df.isRu ? '<span class="badge-ru">RU</span>' : ''}}
        </div>
        ${{!hasPeticion ? `
        <div class="card-count">
          ${{total.toLocaleString('es-ES')}}<span>filas</span>
        </div>
        <div class="bar-track">
          <div class="bar-fill" style="width:${{pct}}%"></div>
        </div>` : ''}}
      </div>
      <div class="card-body">
        <div class="breakdown" style="border-top:none;padding-top:0">
          ${{sinPeticionLabel}}
          <div class="brow" style="color:var(--muted);font-size:.65rem;padding-bottom:.25rem;border-bottom:1px solid var(--border)">
            <span>PRODUCT</span><span>NÚCLEO</span><span class="cnt">#</span>
          </div>
          ${{mainRowsHtml || '<span style="color:var(--muted)">sin datos</span>'}}
        </div>
        ${{opBlock}}
        ${{ruRefBlock}}
      </div>`;
    grid.appendChild(card);
  }});

  if (dfCount === 0) {{
    grid.innerHTML = '<div class="no-data">No hay datos para los filtros seleccionados.</div>';
  }}

  document.getElementById('totalRows').textContent = grandTotal.toLocaleString('es-ES');
  document.getElementById('totalDfs').textContent  = dfCount;
}}

document.getElementById('ts').textContent = new Date().toLocaleString('es-ES');
render();
</script>
</body>
</html>"""

# ─────────────────────────────────────────────
# 4.  GUARDAR
# ─────────────────────────────────────────────
fname = Path(r"C:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\supresiones.html")
with open(fname, "w", encoding="utf-8") as f:
    f.write(HTML)

print(f"✅  HTML generado → {{fname}}")